# 수업 내용
1. 랭체인 기본코드 복습
2. Gradio UI 사용 방법 이해
3. 랭체인 + Gradio UI
4. UI 꾸미는 방법 소개
5. [미션] 나만의 RAG 시스템 만들기


In [ ]:
!pip install -U langchain
!pip install -U langchain-core
!pip install -U langchain-community
!pip install -U langchain-upstage
!pip install -U langgraph-checkpoint-sqlite
!pip install langchain-text-splitters
!pip install tiktoken
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.1/467.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.6/207.6 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
!pip install gradio

# 0-1. 랭체인 기본코드 복습
- Gradio 수업 전, 간단히 랭체인을 복습

In [ ]:
%%capture
!pip install -U langchain 
!pip install -U langchain-upstage

from dotenv import load_dotenv
import os
# from google.colab import userdata

base_path = './'
load_dotenv(base_path + ".env")
os.environ["OPENAI_API_KEY"] = os.getenv("UPSTAGE_API_KEY")


### 0-1-1 랭체인 Agent 기본 코드 실행
- 공식문서의 기본 코드 예제 활용
- 코드 출처 : https://docs.langchain.com/oss/python/langchain/quickstart

In [ ]:
from langchain.agents import create_agent
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """city의 날씨를 알려주는 도구"""
    return "맑음"

agent = create_agent(
    model=ChatUpstage(),
    tools=[get_weather],
    system_prompt="""
      넌 아주 친절한 AI 도구야, 아이처럼 말해
      사용자가 요청하면 필요한 도구를 한번 만 사용해
      넌 get_weather(city)라는 도구를 가지고 있고, 원하는 city의 날씨를 반환해주는 도구지
    """
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "오늘 구미 날씨가 궁금해"}]}
)

answer = result["messages"][-1].content
print(answer)

# 0-2. Gradio UI

- 기본 코드를 실행하기 전에, 아래 코드를 실행
  - 코랩에서 재실행 속도를 높혀주는 확장도구

In [ ]:
%load_ext gradio

- Gradio 공식 Hello world 코드
- 강도 스크롤을 높일수록 "!" 가 증가
  - 코드 출처 : https://www.gradio.app/guides/quickstart#building-your-first-demo
- 코드 실행시 제공되는 URL은 1주일간 무료제공

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
)

demo.launch(share=True)

# 0-3. 랭체인과 Gradio 연동

In [ ]:
import gradio as gr

def chat(message, history):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]})
    answer = result["messages"][-1].content

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return "", history

with gr.Blocks(title="LangChain Agent Chat") as demo:
    gr.Markdown("## 🤖 LangChain Agent Chat\nLangChain Agent가 도구를 활용해 답변합니다.")

    chatbot = gr.Chatbot(
        height=280,
        label="대화창",
        type="messages"
    )

    msg = gr.Textbox(
        label="💬 질문을 입력하세요",
        placeholder="예: 오늘 서울 날씨 어때?",
    )

    # 엔터 누르면 chat 함수 호출
    msg.submit(chat, inputs=[msg, chatbot], outputs=[msg, chatbot])

demo.launch(share=True)


# 1. Agent Mokup

In [ ]:
from google.colab import drive

# 1. Google Drive 마운트
drive.mount('/content/drive')


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/AI/12_gradio/' )

In [ ]:
import gradio as gr

# 1. 간단한 함수 정의
#    (message: 사용자 입력, history: 대화 기록) -> 반환값: 봇 응답
def chat(message, history):
    return f"사용자 입력: {message}"

# 2. ChatInterface에 함수 연결 및 실행
gr.ChatInterface(chat,
                title="AI 온라인 서점 챗봇",
                description="무엇이든 물어보세요!",).launch()

In [ ]:

base_path = "/content/drive/MyDrive/Colab Notebooks/AI/12_gradio/"
!echo "UPSTAGE_API_KEY={upstage_api_key}" > "{base_path}.env"

In [ ]:
from dotenv import load_dotenv
from os import getenv

from langchain_community.vectorstores import Chroma
from langchain_classic.tools.retriever import create_retriever_tool
from langchain.tools import tool

from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime
from typing import Any

from langchain_upstage import ChatUpstage, UpstageEmbeddings
import sqlite3


load_dotenv(base_path + ".env")
UPSTAGE_API_KEY = getenv("UPSTAGE_API_KEY")

embeddings = UpstageEmbeddings(model="embedding-query")

vectorstore = Chroma(
    persist_directory=base_path + "chroma_db_v1",
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever()

retriever_tool = create_retriever_tool(
    retriever,
    "Policy_Search",
    """배송 정책, 보상 정책, 환불 정책 등 AI 온라인 서점의
    다양한 정책 문서에서 고객 질문과 관련된 정보를 검색합니다.
    (예: "배송 지연 시 보상 정책이 어떻게 되나요?")
    """,
)

@tool
def get_order_status_from_db(order_id: str) -> str:
    """(DB 조회)
        주문 ID를 받아 현재 '배송 상태' 문자열을 반환
        단, DB 연결 오류 시 "DB Error: {오류 메시지}" 반환
        주문 ID가 없을 시 "Order Not Found" 반환
    """
    # 디버깅용
    print(f"get_order_status_from_db 호출됨: {order_id} ---")
    try:
        conn = sqlite3.connect(base_path + 'orders.db') # DB에 연결
        c = conn.cursor()                   # 커서: SQL 실행 도구
        # 오직 'status'만 조회
        c.execute("SELECT status FROM orders WHERE order_id = ?", (order_id,))
        result = c.fetchone() # (예: ('Delivered',))
        conn.close()

        if result:
            return result[0] # (예: 'Delivered')
        else:
            return "Order Not Found"
    except Exception as e:
        return f"DB Error: {str(e)}"

# 3. issue_complaint_coupon 함수
@tool
def issue_complaint_coupon(order_id: str):
    """(DB 조회 기반)
        'Shipping Delayed' 상태일 때만 쿠폰을 발급.
        단, 쿠폰 발급 외의 복잡한 로직은 하드코딩으로 처리.
        예를 들어, 쿠폰을 생성하기 위해서는 실제로 쿠폰 테이블에 삽입하거나,
        외부 시스템과 연동하는 등의 작업이 필요하지만, 여기서는 단순히
        주문 상태를 조회하고 조건에 맞으면 쿠폰 발급 메시지를 반환하는 것으로 대체
    """
    # 디버깅용
    print(f"issue_complaint_coupon 호출됨: {order_id} ---")
    order_status = get_order_status_from_db.invoke(order_id)

    # 만약, 배송 지연 상태라면 쿠폰 발급
    if order_status == "Shipping Delayed":
        return f"주문 ID {order_id}에 대해 5,000원 할인 쿠폰이 발급되었습니다."
    else:
        return f"주문 ID {order_id}는 쿠폰 발급 대상이 아닙니다. 현재 상태: {order_status}"

@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

config: RunnableConfig = {"configurable": {"thread_id": "2"}}

class CustomerServiceAgent:
    def __init__(self):
        # 1. LLM 생성
        self.llm = ChatUpstage()

        # 2. 도구 정의
        self.tools = [retriever_tool, get_order_status_from_db, issue_complaint_coupon]

        # 3. 시스템 프롬프트
        self.system_prompt = """
            당신은 AI 온라인 서점의 고객 서비스 AI 에이전트입니다.

            [중요 규칙] 각 도구의 사용 조건과 설명을 반드시 따르세요.
            * 'Issue_Complaint_Coupon' 도구는 'Get_Order_Status' 결과가 'Shipping Delayed'일 때만 사용.
            * 주문 상태가 불명확하면, 쿠폰을 발급하기 전에 반드시 'Get_Order_Status'를 먼저 사용.
        """

        # 4. 메모리
        self.agent = create_agent(
          model=self.llm,
          tools=self.tools,
          middleware=[trim_messages],
          checkpointer=InMemorySaver(),
      )

In [ ]:
# Agent 인스턴스를 저장할 변수 (캐시)
agent_instance = None

def get_agent_instance():
    """Agent 인스턴스를 한 번만 생성하고 재사용(캐싱)합니다."""
    global agent_instance # 전역 변수 사용 선언

    if agent_instance is None: # 캐시가 비어있을 때(최초 1회)
        print("Agent 인스턴스 로드 중...")
        agent_instance = CustomerServiceAgent() # 한 번 생성

    # 이미 생성되었다면(None이 아니면) 캐시된 인스턴스 반환
    return agent_instance

In [ ]:
# UI(Blocks)와 연결될 메인 함수
async def process_message(message, chat_history):

    # 1. 사용자 메시지 즉시 UI에 반영
    chat_history = chat_history + [{"role": "user", "content": message},
                                  {"role": "assistant", "content": ""}]
    yield "", chat_history # 첫 번째 yield

    # 2. 캐시된 Agent 가져오기
    agent = get_agent_instance()

    # 3. Agent 실행
    result = agent.agent.invoke({"messages": [{"role": "user", "content": message}]}, config)
    agent_response = result['messages'][-1].content

    # 4. Agent의 최종 응답을 봇 응답 칸에 채워넣기
    chat_history[-1]["content"] = agent_response

    # 5. 최종 챗봇 내역을 UI에 반영
    yield gr.update(value=""), chat_history # 두 번째 yield

In [ ]:
with gr.Blocks(title="AI 온라인 서점 챗봇") as demo:
    gr.Markdown("# AI 온라인 서점 챗봇")
    gr.Markdown("무엇이든 물어보세요!")

    chatbot = gr.Chatbot(height=400, type='messages', label="Chat History") # 챗봇 메시지 표시 영역 설정

    with gr.Row():
        textbox = gr.Textbox(placeholder="메시지를 입력하세요...", container=False, scale=7, label="Your Message") # 사용자 입력창 설정
        submit_btn = gr.Button("Submit") # Add a submit button

    submit_btn.click(
        process_message,
        inputs=[textbox, chatbot],
        outputs=[textbox, chatbot]
    )
    textbox.submit(
        process_message,
        inputs=[textbox, chatbot],
        outputs=[textbox, chatbot]
    )

In [ ]:
demo.launch(share=True, inline=True, debug = True)

In [ ]:
demo.close()